In [25]:
# Importations
import os
import sys
import uuid
import logging
import psycopg
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, FloatType, DoubleType
)
from pyspark.sql.functions import (
    col, to_timestamp, concat_ws, lit, current_timestamp, regexp_replace, to_date, date_format
)


In [26]:
# Chargement des variables d'environnement (.env)
load_dotenv()


True

In [27]:
# Paramètres de connexion et chemins
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DATASETS_DIR = os.getenv("DATASETS_DIR")

JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}


In [ ]:
# Choix du fichier CSV à ingérer
csv_filename = "20240615_wind_power_data.csv"  
csv_path = os.path.join(DATASETS_DIR, csv_filename)


In [29]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WindPowerBronzeIngestion") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()


In [30]:
# Définition du schéma du CSV
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DoubleType

csv_schema = StructType([
    StructField("production_id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("time", StringType(), True),
    StructField("turbine_name", StringType(), True),
    StructField("capacity", IntegerType(), True),
    StructField("location_name", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("region", StringType(), True),
    StructField("status", StringType(), True),
    StructField("responsible_department", StringType(), True),
    StructField("wind_speed", FloatType(), True),
    StructField("wind_direction", StringType(), True),
    StructField("energy_produced", FloatType(), True)
])


In [31]:
# Lecture du CSV dans un DataFrame Spark
df_raw = spark.read \
    .option("header", "true") \
    .schema(csv_schema) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .csv(csv_path)
df_raw.show(5)

+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+----------+----------------------+----------+--------------+---------------+
|production_id|      date|    time|turbine_name|capacity|location_name|latitude|longitude|  region|    status|responsible_department|wind_speed|wind_direction|energy_produced|
+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+----------+----------------------+----------+--------------+---------------+
|         6049|2024-06-15|00-00-00|   Turbine A|    2200|   Location 1| 34.0522|-118.2437|Region A|    Online|            Operations|   5.74042|            SW|      1783.3937|
|         6050|2024-06-15|00-00-00|   Turbine B|    2000|   Location 2| 36.7783|-119.4179|Region B|Inspection|           Maintenance|  23.92376|             E|            0.0|
|         6051|2024-06-15|00-00-00|   Turbine C|    2500|   Location 3| 40.7128|  -74.006|Region C|    Online|          

In [32]:
# Ajout des colonnes de métadonnées et transformation
df_bronze = (
    df_raw
    .withColumn(
        "measured_at",
        to_timestamp(
            concat_ws(" ", col("date"), regexp_replace(col("time"), lit("-"), lit(":")))
        )
    )
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    .withColumn("time", date_format(to_timestamp(col("time"), "HH-mm-ss"), "HH:mm:ss"))
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", lit(os.path.basename(csv_path)))
    .withColumn("batch_id", lit(str(uuid.uuid4())))
)
df_bronze.show(5)

+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+----------+----------------------+----------+--------------+---------------+-------------------+--------------------+--------------------+--------------------+
|production_id|      date|    time|turbine_name|capacity|location_name|latitude|longitude|  region|    status|responsible_department|wind_speed|wind_direction|energy_produced|        measured_at|         ingested_at|         source_file|            batch_id|
+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+----------+----------------------+----------+--------------+---------------+-------------------+--------------------+--------------------+--------------------+
|         6049|2024-06-15|00:00:00|   Turbine A|    2200|   Location 1| 34.0522|-118.2437|Region A|    Online|            Operations|   5.74042|            SW|      1783.3937|2024-06-15 00:00:00|2026-05-01 02:37:...|2024061

In [33]:
# Sélection des colonnes dans l'ordre attendu
columns_table = [
    "production_id", "date", "time", "turbine_name", "capacity", "location_name", "latitude", "longitude", "region", "status", "responsible_department", "wind_speed", "wind_direction", "energy_produced", "measured_at", "ingested_at", "source_file", "batch_id"
]
df_bronze = df_bronze.select(*columns_table)
df_bronze.show(5)

+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+----------+----------------------+----------+--------------+---------------+-------------------+--------------------+--------------------+--------------------+
|production_id|      date|    time|turbine_name|capacity|location_name|latitude|longitude|  region|    status|responsible_department|wind_speed|wind_direction|energy_produced|        measured_at|         ingested_at|         source_file|            batch_id|
+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+----------+----------------------+----------+--------------+---------------+-------------------+--------------------+--------------------+--------------------+
|         6049|2024-06-15|00:00:00|   Turbine A|    2200|   Location 1| 34.0522|-118.2437|Region A|    Online|            Operations|   5.74042|            SW|      1783.3937|2024-06-15 00:00:00|2026-05-01 02:37:...|2024061

In [34]:
# Création du schéma et de la table dans PostgreSQL (si besoin)
try:
    conn = psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    with conn:
        with conn.cursor() as cur:
            cur.execute("CREATE SCHEMA IF NOT EXISTS bronze")
            # cur.execute("DROP TABLE IF EXISTS bronze.raw_windpower")
            cur.execute("""
                CREATE TABLE IF NOT EXISTS bronze.raw_windpower (
                    production_id INTEGER,
                    date DATE,
                    time TIME,
                    turbine_name VARCHAR(100),
                    capacity INTEGER,
                    location_name VARCHAR(100),
                    latitude DOUBLE PRECISION,
                    longitude DOUBLE PRECISION,
                    region VARCHAR(100),
                    status VARCHAR(50),
                    responsible_department VARCHAR(100),
                    wind_speed FLOAT,
                    wind_direction VARCHAR(10),
                    energy_produced FLOAT,
                    measured_at TIMESTAMP,
                    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    source_file VARCHAR(255),
                    batch_id VARCHAR(100)
                )
            """)
            cur.execute("CREATE INDEX IF NOT EXISTS idx_raw_windpower_turbine_time ON bronze.raw_windpower(turbine_name, measured_at)")
            cur.execute("CREATE INDEX IF NOT EXISTS idx_raw_windpower_location ON bronze.raw_windpower(latitude, longitude)")
            conn.commit()
    print("Schéma et table créés avec succès.")
except Exception as e:
    print(f"Erreur lors de la création du schéma/table : {e}")
    raise

Schéma et table créés avec succès.


In [35]:
# Insertion des données dans PostgreSQL via JDBC
df_bronze.write \
    .format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", "bronze.raw_windpower") \
    .option("stringtype", "unspecified") \
    .options(**JDBC_PROPS) \
    .mode("append") \
    .save()
print(f"{df_bronze.count()} lignes insérées dans bronze.raw_windpower")

432 lignes insérées dans bronze.raw_windpower


In [36]:
# Arrêt de la session Spark
spark.stop()